<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
# --- Setup (run once at the top) ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content page (one pseudonymized content item).

There are 30,000 rows and 32 distinct clients.Time window: every metric is aggregated over a trailing 90-day window ending at export time.

Trend fields compare the most recent 30 days vs the previous 30 days (days 31–60 back).

Every page in this slice is at least 90 days old (content_age_days ≥ 90).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows (one row = one page):", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())
print()
print("Duplicate content_ids (should be 0):", df["content_id"].duplicated().sum())
print()
print("content_age_days min / median / max:")
print(df["content_age_days"].agg(["min", "median", "max"]).round(0))
print()
print("All rows have impressions_90d >= 1:", (df["impressions_90d"] >= 1).all())


Rows (one row = one page): 30000
Unique content_id: 30000
Unique client_id: 32

Duplicate content_ids (should be 0): 0

content_age_days min / median / max:
min        90.0
median    236.0
max       564.0
Name: content_age_days, dtype: float64

All rows have impressions_90d >= 1: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (safe to use for ranking — known before the decision):

impressions_90d, clicks_90d, sessions_90d, pageviews_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, word_count, content_age_days, days_since_last_update, search_volume, competition, cpc, content_type, main_intent, position_tier, impression_tier, age_tier, freshness_tier, word_count_tier

Label / proxy (never a feature):

trend_direction, trend_pct → target is is_declining_label = (trend_direction == "down")

Context (grouping / splits / reading only):

content_id, client_id


Excluded (and why):

provider_used, model_used — generation metadata, not a performance signal for ranking
Any product decision flags (not present in this export) — we must not learn FlyRank’s own rules
trend_direction / trend_pct as features — that would leak the label

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Checks below confirm grain, label base rate, and patterned missingness.**

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Grain
print("=== GRAIN ===")
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_ids:", df["content_id"].duplicated().sum())

# Label
print("\n=== LABEL ===")
print(df["is_declining_label"].value_counts())
print("Base rate declining:", round(df["is_declining_label"].mean(), 3))

# Missingness (key fields)
print("\n=== MISSINGNESS (% blank) ===")
check_cols = ["word_count", "search_volume", "competition", "main_intent",
              "ctr", "avg_position", "trend_pct", "sessions_90d"]
for c in check_cols:
    miss = df[c].isna().mean() * 100
    print(f"{c:20s} {miss:5.1f}%")

# Patterned missingness by content_type
print("\n=== word_count missing by content_type ===")
print(df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean()*100, 1)))

print("\n=== search_volume missing by content_type ===")
print(df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean()*100, 1)))


=== GRAIN ===
Rows: 30000
Unique content_id: 30000
Duplicate content_ids: 0

=== LABEL ===
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate declining: 0.542

=== MISSINGNESS (% blank) ===
word_count            25.7%
search_volume          8.2%
competition            8.2%
main_intent            7.9%
ctr                    0.0%
avg_position           0.0%
trend_pct             11.3%
sessions_90d           0.0%

=== word_count missing by content_type ===
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

=== search_volume missing by content_type ===
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4
Name: search_volume, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

##**What this starter slice cannot tell us:**
No true forward window. trend_direction is built from the same 90-day snapshot (last 30 vs prev 30). A production system would define the label on a future window so features never see the outcome period.

Single snapshot, not a panel. We cannot study seasonality or multi-month recovery paths here. That needs the warehouse daily fact table.
Unbalanced / sparse signals. Keyword fields are systematically missing for some content_types (e.g. feedly articles). A blind fillna(0) would encode content type into the model.

No client names, URLs, or queries. Correct for safety, but we cannot diagnose a specific page in the real world from this file alone.
Rate columns are ×100 percentages. ctr = 0.76 means 0.76%, not 76%. Misreading the scale breaks any comparison.

For the capstone we will move to the warehouse release, write a tighter contract with explicit feature/label windows, and keep the same careful language.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows with blank trend_pct (prev_30d impressions = 0):", df["trend_pct"].isna().sum())
print("Content types:", df["content_type"].value_counts().to_dict())


Rows with blank trend_pct (prev_30d impressions = 0): 3388
Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.